<a href="https://colab.research.google.com/github/Huzaifa3049/retain-mimic/blob/main/notebooks/01_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/'
files = os.listdir(BASE_PATH)
for file in files:
    if file.endswith('.csv') or file.endswith('.ipynb'):
        print(file)

Untitled0.ipynb
LLMfromScratch.ipynb
Untitled1.ipynb
Untitled2.ipynb
Untitled3.ipynb
ADMISSIONS.csv
DIAGNOSES_ICD.csv
PATIENTS.csv
01_data_extraction.ipynb


In [ ]:
import pandas as pd
admissions = pd.read_csv(BASE_PATH + 'ADMISSIONS.csv')
diagnoses  = pd.read_csv(BASE_PATH + 'DIAGNOSES_ICD.csv')
patients   = pd.read_csv(BASE_PATH + 'PATIENTS.csv')

print(f"ADMISSIONS:  {admissions.shape}")
print(f"DIAGNOSES:   {diagnoses.shape}")
print(f"PATIENTS:    {patients.shape}")

ADMISSIONS:  (129, 19)
DIAGNOSES:   (1761, 5)
PATIENTS:    (100, 8)


In [ ]:
admissions.head(3)

,row_id,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,religion,marital_status,ethnicity,edregtime,edouttime,diagnosis,hospital_expire_flag,has_chartevents_data
0,12258,10006,142345,2164-10-23 21:09:00,2164-11-01 17:15:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,SEPARATED,BLACK/AFRICAN AMERICAN,2164-10-23 16:43:00,2164-10-23 23:00:00,SEPSIS,0,1
1,12263,10011,105331,2126-08-14 22:32:00,2126-08-28 18:59:00,2126-08-28 18:59:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Private,NaN,CATHOLIC,SINGLE,UNKNOWN/NOT SPECIFIED,NaN,NaN,HEPATITIS B,1,1
2,12265,10013,165520,2125-10-04 23:36:00,2125-10-07 15:13:00,2125-10-07 15:13:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,NaN,UNKNOWN/NOT SPECIFIED,NaN,NaN,SEPSIS,1,1


In [ ]:
diagnoses.head(3)

,row_id,subject_id,hadm_id,seq_num,icd9_code
0,112344,10006,142345,1,99591
1,112345,10006,142345,2,99662
2,112346,10006,142345,3,5672


In [ ]:
patients.head(3)

,row_id,subject_id,gender,dob,dod,dod_hosp,dod_ssn,expire_flag
0,9467,10006,F,2094-03-05 00:00:00,2165-08-12 00:00:00,2165-08-12 00:00:00,2165-08-12 00:00:00,1
1,9472,10011,F,2090-06-05 00:00:00,2126-08-28 00:00:00,2126-08-28 00:00:00,NaN,1
2,9474,10013,F,2038-09-03 00:00:00,2125-10-07 00:00:00,2125-10-07 00:00:00,2125-10-07 00:00:00,1


In [ ]:
hf_codes = diagnoses[diagnoses['icd9_code'].astype(str).str.startswith('428')]
print(f"Heart failure diagnosis records: {len(hf_codes)}")
print(f"Unique patients with HF: {hf_codes['subject_id'].nunique()}")
print(f"\nHF codes found:")
print(hf_codes['icd9_code'].value_counts())

Heart failure diagnosis records: 61
Unique patients with HF: 35

HF codes found:
icd9_code
4280     39
42832     6
42821     4
42833     3
42831     2
42830     2
42822     2
42843     2
42823     1
Name: count, dtype: int64


In [ ]:
visits_per_patient = admissions.groupby('subject_id')['hadm_id'].count()
print(visits_per_patient.value_counts().sort_index())
print(f"\nMean visits per patient: {visits_per_patient.mean():.2f}")
print(f"Max visits: {visits_per_patient.max()}")
print(f"Min visits: {visits_per_patient.min()}")


print(f"\nPatients with >1 visit: {(visits_per_patient > 1).sum()}")
print(f"Patients with only 1 visit: {(visits_per_patient == 1).sum()}")

hadm_id
1     86
2     11
3      2
15     1
Name: count, dtype: int64

Mean visits per patient: 1.29
Max visits: 15
Min visits: 1

Patients with >1 visit: 14
Patients with only 1 visit: 86


The demo dataset contains only 14 patients with mutliple visits, meaning that the application of the attention mechanisim is difficult to implement and would produce little details about the improvement we get from alpha and beta interpretabilty however it is still good at explaining the architecture ( i hope so)

In [ ]:
import numpy as np
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions = admissions.sort_values(['subject_id', 'admittime'])

codes = diagnoses['icd9_code'].astype(str).unique().tolist()
codes.sort()

code_to_idx = {code: idx+1 for idx, code in enumerate(codes)}
idx_to_code = {idx+1: code for idx, code in enumerate(codes)}
vocab_size   = len(code_to_idx) + 1

In [ ]:
print(f"Vocabulary size (unique ICD codes): {len(code_to_idx)}")
print(f"Sample codes: {codes[:5]}")


Vocabulary size (unique ICD codes): 581
Sample codes: ['00845', '0380', '03811', '03812', '03819']


In [ ]:
visit_codes = diagnoses.groupby('hadm_id')['icd9_code'].apply(
    lambda x : [code_to_idx[str(code)] for code in x]
).to_dict()

Building the patient sequence , each pateint becomes a list of visit, each visit is a list of the disaese codes

In [ ]:
admissions.head()


,row_id,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,religion,marital_status,ethnicity,edregtime,edouttime,diagnosis,hospital_expire_flag,has_chartevents_data
0,12258,10006,142345,2164-10-23 21:09:00,2164-11-01 17:15:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,SEPARATED,BLACK/AFRICAN AMERICAN,2164-10-23 16:43:00,2164-10-23 23:00:00,SEPSIS,0,1
1,12263,10011,105331,2126-08-14 22:32:00,2126-08-28 18:59:00,2126-08-28 18:59:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Private,NaN,CATHOLIC,SINGLE,UNKNOWN/NOT SPECIFIED,NaN,NaN,HEPATITIS B,1,1
2,12265,10013,165520,2125-10-04 23:36:00,2125-10-07 15:13:00,2125-10-07 15:13:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,NaN,UNKNOWN/NOT SPECIFIED,NaN,NaN,SEPSIS,1,1
3,12269,10017,199207,2149-05-26 17:19:00,2149-06-03 18:42:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,SNF,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,2149-05-26 12:08:00,2149-05-26 19:45:00,HUMERAL FRACTURE,0,1
4,12270,10019,177759,2163-05-14 20:43:00,2163-05-15 12:00:00,2163-05-15 12:00:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,NaN,NaN,ALCOHOLIC HEPATITIS,1,1


In [ ]:
patient_sequences = {}

for subject_id , group in admissions.groupby('subject_id'):
    visits = []
    for hadm_id in group['hadm_id']:
        codes = visit_codes.get(hadm_id, [])
        if len(codes) > 0:
            visits.append(codes)
    if len(visits) > 0:
      patient_sequences[subject_id] = visits



In [ ]:
print(len(patient_sequences))

100


In [ ]:
print(list(patient_sequences.keys())[0])

10006


In [ ]:
example_id = list(patient_sequences.keys())[0]
example_seq = patient_sequences[example_id]
print(f"\nExample patient {example_id}:")
print(f"  Number of visits: {len(example_seq)}")
for i, visit in enumerate(example_seq):
    codes_readable = [idx_to_code[idx] for idx in visit]
    print(f"  Visit {i+1}: {codes_readable}")



Example patient 10006:
  Number of visits: 1
  Visit 1: ['99591', '99662', '5672', '40391', '42731', '4280', '4241', '4240', '2874', '03819', '7850', 'E8791', 'V090', '56211', '28529', '25000', 'V5867', 'E9342', '41401', '2749', '3051']


Building labels and getting heart failure patients

In [ ]:
hf_patients = set(diagnoses [
    diagnoses['icd9_code'].astype(str).str.startswith('428')
]['subject_id'].tolist())

In [ ]:
print(len(hf_patients))

35


In [ ]:
labels = {}
for subject_id in patient_sequences.keys():
  labels[subject_id] = 1 if subject_id in hf_patients else 0

In [ ]:
total     = len(labels)
positive  = sum(labels.values())
negative  = total - positive


In [ ]:
print(total)
print(positive)
print(negative)

100
35
65


In [ ]:
import pickle

data = {
    'patient_sequence': patient_sequences,
    'labels' : labels,
    'code_to_idx' : code_to_idx ,
    'idx_to_code' : idx_to_code,
    'vocab_size' : vocab_size
}

In [ ]:
save_path = BASE_PATH + 'retain_data.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(data, f)
